# 03 · Stress ramp: does my cluster scale up, and back down?

**Who this is for:** anyone who needs to *see* the platform's elasticity rather than be told about
it. The notebook submits a ramp of checkmaite runs to your cluster — more work than the head can
do alone — and samples the cluster's shape every few seconds while it does: how many nodes are
alive, how many CPUs Ray offers, how many are free, how many jobs are done. Then it waits for the
workers to be let go again.

What you should see with the `checkmaite` profile (workers `min 0 / max 2`, autoscaling on):
nodes go **1 → 3** within a minute of the ramp, back to **1** a minute or two after the last run.
If the control plane runs without autoscaling, the worker count stays at the profile's replicas
and the summary says so — that is a finding, not a failure of this notebook.

Knobs: `STRESS_RUNS` (default 12), `STRESS_CPUS_PER_RUN` (default 2), `STRESS_CONCURRENCY`
(runs in flight at once, default 3 — see the ramp cell for why), `SCALE_DOWN_WAIT_S` (default 600).

Run `01-my-cluster.ipynb` first; `02-checkmaite-capability.ipynb` explains the checkmaite half.

In [ ]:
# The Bifrost sidebar talks to a small server extension inside this very
# JupyterLab (`/user/<you>/bifrost/*`). A notebook can call the same routes
# with the server's own hub token, so what happens here is exactly what a
# click in the sidebar does: same identity, same project, same NetworkPolicy.
import os, time, json, requests

SERVER = os.environ["JUPYTERHUB_SERVICE_URL"]          # http://0.0.0.0:8888/user/<you>/
USER = os.environ.get("JUPYTERHUB_USER", "me")
_HDR = {"Authorization": f"token {os.environ['JUPYTERHUB_API_TOKEN']}"}

def ext(method, path, body=None, **kw):
    """Call an extension route; returns (status, json-or-text)."""
    r = requests.request(method, SERVER + "bifrost/" + path, headers=_HDR, json=body, timeout=60, **kw)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

def my_running_cluster():
    """The cluster these notebooks share: BIFROST_CLUSTER_ID if set, else the one running cluster."""
    want = os.environ.get("BIFROST_CLUSTER_ID")
    status, view = ext("GET", "clusters")
    assert status == 200 and view.get("configured", True), f"extension not configured: {status} {view}"
    running = [c for c in view["clusters"] if c["state"] == "running"]
    if want:
        return next((c for c in running if c["id"] == want), None)
    return running[0] if len(running) == 1 else None

def wait_running(cluster_id, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        status, view = ext("GET", "clusters")
        state = next((c["state"] for c in view["clusters"] if c["id"] == cluster_id), "gone")
        print(f"{time.time()-t0:5.0f}s  {cluster_id}: {state}", flush=True)
        if state == "running":
            return
        if state in ("failed", "terminated", "gone"):
            raise RuntimeError(f"cluster {cluster_id} went {state}")
        time.sleep(10)
    raise TimeoutError(f"cluster {cluster_id} not running after {timeout}s")

print("notebook user:", USER, "| server:", SERVER)

In [ ]:
import ray
from bifrost_jupyter._address import ray_client_address
from bifrost_jupyter.config import default_namespace

cluster = my_running_cluster()
assert cluster, "no running cluster: run 01-my-cluster.ipynb first (or set BIFROST_CLUSTER_ID)"
CLUSTER_ID = cluster["id"]
RAY_ADDRESS = ray_client_address(CLUSTER_ID, default_namespace())
print("cluster", CLUSTER_ID, "->", RAY_ADDRESS)

In [ ]:
# What does the cluster have? Same checkmaite/ray/python as this kernel is the
# contract for driving it from here; the analytics volume is the profile's storage.
if ray.is_initialized():
    ray.shutdown()
ray.init(RAY_ADDRESS, logging_level="ERROR")

@ray.remote(num_cpus=0)
def probe(paths):
    import sys, os, ray as _r, checkmaite as _c
    return {
        "python": sys.version.split()[0], "ray": _r.__version__, "checkmaite": _c.__version__,
        "mounted": {p: os.path.isdir(p) for p in paths},
        "node": _r.get_runtime_context().get_node_id()[:8],
    }

import checkmaite, sys
DATA_ROOT = os.environ.get("CHECKMAITE_DATA_ROOT", "/app/data/analytics")
remote = ray.get(probe.remote([DATA_ROOT, f"{DATA_ROOT}/datasets"]))
local = {"python": sys.version.split()[0], "ray": ray.__version__, "checkmaite": checkmaite.__version__}
print("notebook:", local)
print("cluster :", remote)
for k in local:
    assert local[k].split(".")[:2] == remote[k].split(".")[:2], f"{k} differs: {local[k]} vs {remote[k]}"
ANALYTICS_MOUNTED = remote["mounted"][DATA_ROOT]
print("analytics volume on the cluster:", ANALYTICS_MOUNTED)
# The connection stays open: with the volume mounted no runtime_env is needed and
# the job backend below simply reuses it.

In [ ]:
# The dataset is opened where the files are. With the analytics volume mounted the
# demo datasets are on every node and the dataset object is built by a task on the
# cluster (its files never come to the notebook). Without it, a small synthetic
# YOLO-classification set is generated here and shipped with the runtime_env, so
# the notebook still works — slower and ephemeral, but it works.
DATASET_NAME = os.environ.get("CHECKMAITE_DATASET", "demo-ic-baseline")
runtime_env = {}

if ANALYTICS_MOUNTED:
    DATA_DIR = f"{DATA_ROOT}/datasets/{DATASET_NAME}"
    STORE_URI = f"{DATA_ROOT}/notebooks/{USER}"

    def build_dataset(root, split, dataset_id=None):
        @ray.remote(num_cpus=0)
        def _build(root, split, dataset_id):
            from checkmaite.core.image_classification.dataset_loaders import YoloClassificationDataset
            ds = YoloClassificationDataset(root, split=split, dataset_id=dataset_id)
            return ds, len(ds)
        return ray.get(_build.remote(root, split, dataset_id))
else:
    import pathlib, random
    from PIL import Image, ImageDraw
    local_root = pathlib.Path("synthetic-ic"); DATA_DIR = "synthetic-ic"
    for label in ("square", "circle"):
        d = local_root / "val" / label; d.mkdir(parents=True, exist_ok=True)
        for i in range(12):
            im = Image.new("RGB", (64, 64), (random.randint(180, 255),) * 3); dr = ImageDraw.Draw(im)
            box = (random.randint(4, 20), random.randint(4, 20), random.randint(40, 60), random.randint(40, 60))
            (dr.rectangle if label == "square" else dr.ellipse)(box, fill=(random.randint(0, 120), 0, random.randint(0, 120)))
            im.save(d / f"{label}_{i:02d}.png")
    runtime_env = {"working_dir": str(local_root.parent.resolve()), "excludes": ["*.ipynb", ".ipynb_checkpoints"]}
    STORE_URI = "/tmp/checkmaite-analytics"   # on the cluster; gone with it
    print("no analytics volume on the cluster -> synthetic dataset shipped via runtime_env")

    def build_dataset(root, split, dataset_id=None):
        from checkmaite.core.image_classification.dataset_loaders import YoloClassificationDataset
        ds = YoloClassificationDataset(root, split=split, dataset_id=dataset_id)
        return ds, len(ds)

print("dataset dir:", DATA_DIR, "| analytics store:", STORE_URI)

## The ramp

Each run is a `DataevalCleaning` over the same images but under its own `dataset_id`, so
checkmaite's idempotency does not collapse them into one. Each asks Ray for
`STRESS_CPUS_PER_RUN` CPUs — **2 by default, the size of a worker in the `checkmaite`
profile**. The head has 2 CPUs and its registry/controller actors already hold a sliver of
them, so a 2-CPU run cannot fit on the head at all: every run sits *pending* until a worker
exists, and pending resource demand is precisely what the autoscaler acts on. (With 1 CPU per
run the demo dataset is small enough that runs finish on the head in a few seconds each and
nothing is ever pending long enough to scale; and eight of them at once on an 8 Gi head is
enough for Ray's memory monitor to kill a couple — a finding in its own right, recorded in
the outcomes below.)

In [ ]:
from checkmaite.jobs import configure_job_backend, submit_capability
from checkmaite.core.image_classification import DataevalCleaning

RUNS = int(os.environ.get("STRESS_RUNS", 12))
CPUS = float(os.environ.get("STRESS_CPUS_PER_RUN", 2))
IN_FLIGHT = int(os.environ.get("STRESS_CONCURRENCY", 3))
SCALE_DOWN_WAIT_S = int(os.environ.get("SCALE_DOWN_WAIT_S", 600))


def connect_backend(**backend_kwargs):
    """Point checkmaite's job backend at the cluster, reconnecting only when the runtime_env needs it.

    Reconnecting means the head's Ray Client proxier forks a fresh server for us; that
    fork occasionally dies at birth (a gRPC check failure) when connections come in
    quick succession, and the client sees ConnectionAbortedError — so a reconnect is
    retried a few times, which is what a person would do.
    """
    reconnect = bool(runtime_env) or not ray.is_initialized()
    for attempt in range(1, 5):
        try:
            configure_job_backend(
                "ray",
                address=RAY_ADDRESS if reconnect else None,
                runtime_env=runtime_env or None,
                analytics_store={"backend": "parquet", "uri": STORE_URI},
                force_reinit=reconnect,
                # checkmaite keeps one controller actor per job on the head, ~0.9 GiB each
                # (it imports torch), for an hour after the job ends by default. On an
                # 8 GiB head that is the real ceiling on concurrent runs, so let them go.
                controller_retention_s=30,
                max_retained_terminal_controllers=2,
                **backend_kwargs,
            )
            return
        except ConnectionAbortedError as exc:
            print(f"Ray Client server did not start (attempt {attempt}): {str(exc).strip().splitlines()[-1]}")
            time.sleep(5 * attempt)
    raise RuntimeError("could not connect checkmaite's job backend to the cluster")

# One scope per user: the registry actor behind a scope costs as much as a controller.
connect_backend(idempotency_scope=f"notebook-{USER}")

def snapshot(phase, jobs):
    alive = [n for n in ray.nodes() if n["Alive"]]
    res, avail = ray.cluster_resources(), ray.available_resources()
    done = sum(1 for j in jobs if j.status.name in ("COMPLETED", "FAILED", "CANCELLED"))
    return {"t": round(time.time() - T0, 1), "phase": phase, "nodes": len(alive),
            "workers": max(0, len(alive) - 1), "cpu_total": res.get("CPU", 0), "cpu_free": avail.get("CPU", 0),
            "jobs_done": done, "jobs_total": len(jobs)}

T0 = time.time()
samples, jobs = [], []
samples.append(snapshot("idle", jobs))
baseline_workers = samples[0]["workers"]
print("baseline:", samples[0])

def in_flight(jobs):
    return sum(1 for j in jobs if j.status.name not in ("COMPLETED", "FAILED", "CANCELLED"))

RUN_TAG = int(time.time())
for i in range(RUNS):
    # At most IN_FLIGHT runs outstanding: each one is a controller actor on the head.
    while in_flight(jobs) >= IN_FLIGHT:
        samples.append(snapshot("ramp", jobs)); time.sleep(3)
    ds, _ = build_dataset(DATA_DIR, "val", dataset_id=f"stress-{USER}-{RUN_TAG}-{i:02d}")
    jobs.append(submit_capability(DataevalCleaning(), datasets=[ds], resources={"num_cpus": CPUS}))
    samples.append(snapshot("ramp", jobs))
    print(f'{samples[-1]["t"]:6.1f}s submitted {i+1}/{RUNS}  nodes={samples[-1]["nodes"]} cpu_free={samples[-1]["cpu_free"]}  in flight={in_flight(jobs)}', flush=True)

## Watch it work

In [ ]:
while True:
    s = snapshot("ramp", jobs); samples.append(s)
    print(f'{s["t"]:6.1f}s nodes={s["nodes"]} workers={s["workers"]} cpu={s["cpu_free"]:.0f}/{s["cpu_total"]:.0f} free  done {s["jobs_done"]}/{s["jobs_total"]}', flush=True)
    if s["jobs_done"] == s["jobs_total"]:
        break
    if s["t"] > 1800:
        raise TimeoutError("ramp did not finish in 30 minutes")
    time.sleep(5)

outcomes = {}
for j in jobs:
    st = j.status.name
    outcomes[st] = outcomes.get(st, 0) + 1
    if st != "COMPLETED":
        print("job", j.job_id, st, j.exception())
print("outcomes:", outcomes)
RAMP_END = time.time() - T0

## …and let go

With nothing pending, the autoscaler removes idle workers after its idle timeout (KubeRay's
default is 60 s). Sampling continues until the worker count is back to the baseline or the
wait runs out.

In [ ]:
peak = max(s["workers"] for s in samples)
while True:
    s = snapshot("drain", jobs); samples.append(s)
    print(f'{s["t"]:6.1f}s nodes={s["nodes"]} workers={s["workers"]}', flush=True)
    if s["workers"] <= baseline_workers or (time.time() - T0 - RAMP_END) > SCALE_DOWN_WAIT_S:
        break
    time.sleep(10)

## What happened

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(samples)
first_up = df[(df.phase == "ramp") & (df.workers > baseline_workers)].t.min()
ramp_rows = df[df.phase == "ramp"]
last_at_peak = df[df.workers == peak].t.max()
back_down = df[(df.phase == "drain") & (df.workers <= baseline_workers)].t.min()

summary = {
    "runs": RUNS, "cpus_per_run": CPUS, "outcomes": outcomes,
    "baseline_workers": baseline_workers, "peak_workers": int(peak),
    "first_scale_up_s": None if pd.isna(first_up) else float(first_up),
    "ramp_finished_s": round(RAMP_END, 1),
    "scaled_back_down_s": None if pd.isna(back_down) else float(back_down),
    "scale_down_after_ramp_s": None if pd.isna(back_down) else round(float(back_down) - RAMP_END, 1),
    "autoscaled": bool(peak > baseline_workers),
}
print(json.dumps(summary, indent=1))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.step(df.t, df.workers, where="post", label="workers", color="tab:blue")
ax1.set_ylabel("workers"); ax1.set_xlabel("seconds"); ax1.set_ylim(bottom=-0.1)
ax2 = ax1.twinx()
ax2.plot(df.t, df.jobs_done, label="runs finished", color="tab:green")
ax2.plot(df.t, df.cpu_total - df.cpu_free, label="CPUs in use", color="tab:orange", alpha=.7)
ax2.set_ylabel("runs / CPUs")
for ph, colour in (("ramp", "#ffe8b0"), ("drain", "#d8ecff")):
    r = df[df.phase == ph]
    if len(r): ax1.axvspan(r.t.min(), r.t.max(), color=colour, alpha=.5, lw=0)
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.98))
plt.title(f"{CLUSTER_ID}: {RUNS} checkmaite runs — workers {baseline_workers}→{int(peak)}→{int(df.workers.iloc[-1])}")
plt.tight_layout(); plt.show()

# Persisted for reports: the samples and the summary, beside this notebook.
pd.DataFrame(samples).to_csv(f"stress-{CLUSTER_ID}.csv", index=False)
json.dump(summary, open(f"stress-{CLUSTER_ID}.json", "w"), indent=1)

## The claims

Every run must have succeeded. Whether the cluster *scaled* is asserted only when the control
plane says it should have: `EXPECT_AUTOSCALE=1` makes the up-then-down shape a hard requirement;
otherwise the summary's `autoscaled` field records what happened.

In [ ]:
assert outcomes.get("COMPLETED", 0) == RUNS, f"not every run completed: {outcomes}"
if os.environ.get("EXPECT_AUTOSCALE") == "1":
    assert summary["autoscaled"], f"workers never rose above {baseline_workers} under {RUNS} runs"
    assert summary["scaled_back_down_s"] is not None, f"workers did not return to {baseline_workers} within {SCALE_DOWN_WAIT_S}s of the ramp"
print("ok:", "scaled up and back down" if summary["autoscaled"] and summary["scaled_back_down_s"] else "fixed-size cluster, all runs succeeded")
ray.shutdown()